In [41]:
import pandas as pd
import geopandas as gpd
import numpy as np

from tqdm.notebook import tqdm
import requests, cma

import pickle, os

In [42]:
# reference_path = "../../../results/road/freeflow/reference_api.parquet"
# output_path = "../../../results/road/freeflow/calibration_cache_api.pickle"

reference_path = "../../../results/road/freeflow/reference_survey.parquet"
output_path = "../../../results/road/freeflow/calibration_cache_survey_only_majors.pickle"

# routing_endpoint = "http://sma.univ-eiffel.fr:18054/router/road"
routing_endpoint = "http://localhost:8054/router/road"
departure_time = 4 * 3600
maximum_batch_size = 400

In [43]:
# Load reference
df_reference = pd.read_parquet(reference_path)

In [44]:
# Prepare requests
df_reference["request_index"] = np.arange(len(df_reference))

# Convert to requests
request_list = []

for index, row in df_reference.iterrows():
    request_list.append({
        "request_index": int(row["request_index"]),
        "origin_x": row["origin_x"],
        "origin_y": row["origin_y"],
        "destination_x": row["destination_x"],
        "destination_y": row["destination_y"],
        "departure_time_s": departure_time,
        "consider_parallel_links": True
    })

In [45]:
# Prepare querying
def query_requests(request_list, settings):
    df_response = []
    batch_index = 0

    while batch_index * maximum_batch_size < len(request_list):
        batch = request_list[batch_index * maximum_batch_size : (batch_index + 1) * maximum_batch_size]

        response = requests.post(routing_endpoint, verify=False, json = {
            "batch": batch,
            "freeflow": settings
        })

        df_response.append(pd.DataFrame.from_records(response.json()))
        batch_index += 1

    return pd.concat(df_response)

In [46]:
# Define calibration variables
variables = [
    { "name": "major_factor", "initial": 1.0, "bounds": (0.99, 1.01) },
    { "name": "intermediate_factor", "initial": 1.0, "bounds": (0.99, 1.01) },
    { "name": "minor_factor", "initial": 1.0, "bounds": (0.99, 1.01) },
    { "name": "major_crossing_penalty_s", "initial": 0.0, "bounds": (0.0, 10.0) },
    { "name": "equal_crossing_penalty_s", "initial": 0.0, "bounds": (0.0, 1.0) },
    { "name": "minor_crossing_penalty_s", "initial": 0.0, "bounds": (0.0, 0.5) },
]

In [47]:
# Extend with index information for CMA-ES evaluation
variables_map = { v["name"]: v for v in variables }

active_index = 0

# First find active variables
for variable in variables:
    variables_map[variable["name"]] = variable
    variable["index"] = active_index
    active_index += 1

In [48]:
# Define the optimization objective
def calculate_objective(df_evaluation, df_reference):
    df_reference = df_reference[["request_index", "reference_travel_time_s", "weight"]].copy()

    df_evaluation = df_evaluation[["request_index", "total_travel_time_min"]].copy()
    df_evaluation["evaluation_travel_time_s"] = df_evaluation["total_travel_time_min"] * 60.0
    df_evaluation = df_evaluation[["request_index", "evaluation_travel_time_s"]]

    df_comparison = pd.merge(df_reference, df_evaluation, on = "request_index")
    df_comparison["difference_s"] = np.abs(df_comparison["evaluation_travel_time_s"] - df_comparison["reference_travel_time_s"])

    if False:
        mean = np.sum(df_comparison["reference_travel_time_s"] * df_comparison["weight"]) / df_comparison["weight"].sum()
        ss_residuals = np.sum(df_comparison["weight"] * df_comparison["difference_s"]**2)
        ss_total = np.sum(df_comparison["weight"] * (df_comparison["reference_travel_time_s"] - mean)**2)
        R2 = 1 - ss_residuals / ss_total
        
        return (
            1 - R2, df_comparison
        )
    
    return (
        np.sum(df_comparison["weight"] * np.abs(df_comparison["difference_s"])) / df_comparison["weight"].sum(), 
        df_comparison
    )

In [49]:
# Prepare function to convert CMA-ES' candidate to freespeed settings
def prepare_settings(values):
    settings = {}
    
    for variable in variables:
        settings[variable["name"]] = values[variable["index"]]
    
    return settings

In [50]:
# Prepare bounds and initial values
initial = []
bounds = [[], []]

for variable in variables:
    initial.append(variable["initial"])
    bounds[0].append(variable["bounds"][0])
    bounds[1].append(variable["bounds"][1])

In [51]:
query_requests(request_list[:5], prepare_settings(initial))


,request_index,in_vehicle_distance_km,in_vehicle_time_min,access_time_min,egress_time_min,access_distance_km,egress_distance_km,arrivalTime_s,total_travel_time_min
0,0,31.402718,19.336257,1.284642,2.035618,0.071149,0.112742,15637.253913,22.656516
1,1,31.491387,21.002365,0.255208,0.169607,0.014135,0.009394,15675.454393,21.427181
2,2,8.267682,7.055799,2.521721,0.227536,0.139665,0.012602,14974.651243,9.805056
3,3,2.986466,3.302731,1.260304,0.953304,0.069801,0.052798,14673.782109,5.516339
4,4,3.173404,3.491936,0.318554,0.567168,0.017643,0.031412,14628.629401,4.377658


In [52]:
# Test connection
assert len(query_requests(request_list[:5], prepare_settings(initial))) == 5

In [53]:
# Configure CMA-ES
seed = 1000
sigma = 1.0
iterations = 200

options = cma.CMAOptions()
options.set("bounds", bounds)
options.set("seed", seed)

algorithm = cma.CMAEvolutionStrategy(initial, sigma, options)

# Load cached data for previous iterations
history = []

if os.path.exists(output_path):
    with open(output_path, "rb") as f:
        history = pickle.load(f)

        algorithm.feed_for_resume(
            [h["candidate"] for h in history[1:]], # first one is initial
            [h["objective"] for h in history[1:]]
        )

# Perform a new batch of iterations
for iteration in range(iterations):
    initial_evaluation = len(history) == 0
    candidates = [initial]

    if not initial_evaluation:
        candidates = algorithm.ask()

    objectives = []

    for candidate in candidates:
        settings = prepare_settings(candidate)
        df_response = query_requests(request_list, settings)
        objective, df_comparison = calculate_objective(df_response, df_reference)

        objectives.append(objective)

        history.append({
            "candidate": candidate,
            "settings": settings,
            "objective": objective,
            "evaluation": df_comparison,
            "initial": initial_evaluation
        })

    if not initial_evaluation:
        algorithm.tell(candidates, objectives)
        algorithm.disp()

    # Save after a successful CMA-ES iteration
    with open(output_path, "wb+") as f:
        pickle.dump(history, f)

c:\Users\lebescond\AppData\Local\miniforge3\envs\choice_model\Lib\site-packages\cma\evolution_strategy.py:1244: UserWarning: Sampling standard deviation i=0 at iteration 0 change by 0.006666666666666672 to stds[0]=0.006666666666666672
  warnings.warn("Sampling standard deviation i={0} at iteration {1}"
c:\Users\lebescond\AppData\Local\miniforge3\envs\choice_model\Lib\site-packages\cma\evolution_strategy.py:1244: UserWarning: Sampling standard deviation i=1 at iteration 0 change by 0.006666611111342598 to stds[1]=0.006666666666666672
  warnings.warn("Sampling standard deviation i={0} at iteration {1}"
c:\Users\lebescond\AppData\Local\miniforge3\envs\choice_model\Lib\site-packages\cma\evolution_strategy.py:1244: UserWarning: Sampling standard deviation i=2 at iteration 0 change by 0.006666555556481482 to stds[2]=0.006666666666666672
  warnings.warn("Sampling standard deviation i={0} at iteration {1}"
c:\Users\lebescond\AppData\Local\miniforge3\envs\choice_model\Lib\site-packages\cma\evol

(4_w,9)-aCMA-ES (mu_w=2.8,w_1=49%) in dimension 6 (seed=1000, Tue Jul 22 11:49:09 2025)


Iterat #Fevals   function value  axis ratio  sigma  min&max std  t[m:s]
    1      9 4.576882428617993e+02 1.0e+00 1.01e+00  6e-03  1e+00 0:04.9
    2     18 4.539448857110989e+02 1.4e+00 1.15e+00  7e-03  1e+00 0:10.2
    3     27 4.527807528402760e+02 1.6e+00 1.30e+00  7e-03  2e+00 0:15.9
    4     36 4.527494115648163e+02 1.9e+00 1.54e+00  7e-03  2e+00 0:22.0
    5     45 4.528386134133166e+02 2.0e+00 1.57e+00  7e-03  2e+00 0:29.0
    6     54 4.527379173591478e+02 2.1e+00 1.54e+00  6e-03  2e+00 0:35.9
    7     63 4.527367824002709e+02 2.1e+00 1.56e+00  7e-03  2e+00 0:42.7
    9     81 4.525464183146199e+02 2.3e+00 1.46e+00  6e-03  1e+00 0:57.6
   11     99 4.523495923611043e+02 2.3e+00 1.70e+00  7e-03  1e+00 1:11.8
   13    117 4.523518521477813e+02 2.2e+00 1.79e+00  6e-03  1e+00 1:25.9
   15    135 4.524062802862207e+02 2.4e+00 1.71e+00  6e-03  1e+00 1:39.8
   17    153 4.521390988389041e+02 2.5e+00 1.65e+00  5e-03  9e-01 1:54.2
   19    171 4.516539188264668e+02 3.1e+00 1.82e+00 